In [12]:
from typing import Dict

import os
import pickle
from pathlib import Path
import anndata as ad
import pandas as pd
import scanpy as sc
from tabulate import tabulate

import torch
import pytorch_lightning as pl

from vqniche.utils.parse_test_configs import *
from vqniche.initializers.initialize import *
from vqniche import metrics
from vqniche.metrics import compute_mmd_score
from vqniche.utils.type_conversions import *
from vqniche.plotting import *
from vqniche.utils.loss_utils import aggregate_1hop_neighbor_features

In [3]:
def pyg_batch_to_anndata(batch, only_inputs=True):
    to_np = lambda t: t.detach().cpu().numpy()

    if only_inputs:
        idx = np.arange(batch.batch_size)  # or batch.input_id if available
    else:
        idx = slice(None)  # keep all nodes in the sampled subgraph

    # X
    X = to_np(batch.x)[idx]

    # spatial
    spatial = to_np(batch.xy_coordinates)[idx]

    # batch IDs
    batch_ids = to_np(getattr(batch, "adata_batch_ids", np.zeros(len(idx)))).reshape(-1)[idx]

    # y one-hot -> int
    y_onehot = to_np(batch.y)[idx]
    cell_types = y_onehot.argmax(1)

    obs = pd.DataFrame({
        "batch": batch_ids,
        "cell_type": cell_types,
    })
    obs.index = pd.Index([str(i) for i in batch.input_id], name="input_id")

    return ad.AnnData(X=X, obs=obs, obsm={"spatial": spatial})

# mmb0-4b_1p

In [33]:
wandb_run_dir = "/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/mmb0-4b_1p/sweep/VQNiche/batch=[0, 1, 2, 3]/spatial_n_neighs_8/seed/20250918-120551/wandb/run-20250918_120552-y3oljgf1"

config = collect_test_configs(wandb_run_dir = wandb_run_dir)

# --------------------- Determinism Settings ---------------------
pl.seed_everything(config['experiment']['seed'])

# --------------------- Dataset ---------------------
dataset_blob = initialize_dataset_blob(config)
with open(Path(dataset_blob.processed_dir) / 'label_categories.pkl', 'rb') as f:
    label_categories = pickle.load(f)

# --------------------- Databatch ---------------------
data_batch = initialize_databatch(
                config=config,
                dataset_blob=dataset_blob,
            )

# --------------------- Dataloader ---------------------
datamodule_batch = initialize_datamodule(
                        config=config,
                        data=data_batch,

                    )


Seed set to 0


Best checkpoint found: epoch=4-train_pearson_1hop_nbr=0.94.ckpt
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
DataBatch(y_cell_types=[48156, 23], y_niche_types=[48156, 4], xy_coordinates=[48156, 2], cell_id=[4], dataset_id=[4], tissue=[4], species=[4], adata_batch_id=[4], x=[48156, 1000], y=[48156, 23], edge_index=[2, 497936], encoder_condition_dim=0, spatial_prior_feature_dim=0, attr_decoder_condition_dim=0, adj_decoder_condition_dim=0, num_features=1000, num_classes=23, num_nodes=48156, num_edges=[4], train_mask=[48156], val_mask=[48156], test_mask=[48156], batch=[48156], ptr=[5], adata_batch_ids=[48156])
Batch ID(s): [0, 1, 2, 3]
Data Batch: DataBatch(y_cell_types=[48156, 23], y_niche_types=[48156, 4], xy_coordinates=[48156, 2], cell_id=[4], dataset_id=[4], tissue=[4], species=[4], adata_batch_id=[4], x=[48156, 1000], y=[48156, 23], edge_index=[2, 497936]

In [3]:
dataset_blob

InMemoryDatasetBlob(4)

In [34]:
for data in dataset_blob:
    print(data)
    break

SubsetHVG: Subsetted data to 1000 features.
Data(y_cell_types=[9499, 23], y_niche_types=[9499, 4], xy_coordinates=[9499, 2], cell_id=[9499], dataset_id=0, tissue='brain', species='mus_musculus', adata_batch_id=[1], x=[9499, 1000], y=[9499, 23], edge_index=[2, 98585], encoder_condition_dim=0, spatial_prior_feature_dim=0, attr_decoder_condition_dim=0, adj_decoder_condition_dim=0, num_features=1000, num_classes=23, num_nodes=9499, num_edges=98585, train_mask=[9499], val_mask=[9499], test_mask=[9499])


In [44]:
adata_list = []
for batch in datamodule_batch.train_dataloader():
    a = pyg_batch_to_anndata(batch)
    adata_list.append(a)

In [45]:
adata = ad.concat(
    adata_list,
    axis=0,           # stack rows (cells)
    join="outer",
)

In [46]:
adata

AnnData object with n_obs × n_vars = 47313 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [47]:
adata.write("./mmb0-4b_1p_train.h5ad")

In [38]:
adata_list = []
for batch in datamodule_batch.test_dataloader():
    a = pyg_batch_to_anndata(batch)
    adata_list.append(a)

In [39]:
adata = ad.concat(
    adata_list,
    axis=0,           # stack rows (cells)
    join="outer",
)

In [40]:
adata

AnnData object with n_obs × n_vars = 843 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [42]:
adata.write("./mmb0-4b_1p_test.h5ad")

In [62]:
wfm_test_fname = "/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/analysis/notebooks/mmb0-4b_1p_wfm_test.pkl"
with open(wfm_test_fname, "rb") as f:
    adata_wfm_test = pickle.load(f)
adata_wfm_test

/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.7.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


AnnData object with n_obs × n_vars = 843 × 1000
    obs: 'batch', 'cell_type'
    uns: 'log1p', 'pc_transform', 'niche_scale'
    obsm: 'spatial', 'X_pca', 'niche_mat', 'orig_niche_mat', 'wfm_niche_mat'

In [63]:
compute_mmd_score(
            D=[row for row in adata_wfm_test.obsm['orig_niche_mat']],
            D_hat=[row for row in adata_wfm_test.obsm['wfm_niche_mat']],
            method='scipy',
            kernel='l1_gaussian_tv',
        )

0.1357882602715418

In [64]:
compute_mmd_score(
            D=[row for row in adata_wfm_test.obsm['orig_niche_mat']],
            D_hat=[row for row in adata_wfm_test.obsm['wfm_niche_mat']],
            method='scipy',
            kernel='energy',
        )

0.16527956071392436

# xhs1000-39b_1p-oriented-7

In [57]:
wandb_run_dir = "/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhs1000-39b_1p/sweep/VQNiche/batch=[2, 11, 9, 28, 29, 32, 12]/spatial_n_neighs_8/seed/20250923-222431/wandb/offline-run-20250923_222431-mank50dk"

config = collect_test_configs(wandb_run_dir = wandb_run_dir)

# --------------------- Determinism Settings ---------------------
pl.seed_everything(config['experiment']['seed'])

# --------------------- Dataset ---------------------
dataset_blob = initialize_dataset_blob(config)
with open(Path(dataset_blob.processed_dir) / 'label_categories.pkl', 'rb') as f:
    label_categories = pickle.load(f)

# --------------------- Databatch ---------------------
data_batch = initialize_databatch(
                config=config,
                dataset_blob=dataset_blob,
            )

# --------------------- Dataloader ---------------------
datamodule_batch = initialize_datamodule(
                        config=config,
                        data=data_batch,

                    )


Seed set to 0


Best checkpoint found: epoch=4-train_pearson_1hop_nbr=0.87.ckpt
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
DataBatch(y_cell_types=[110278, 41], y_niche_types=[110278, 15], xy_coordinates=[110278, 2], cell_id=[7], dataset_id=[7], tissue=[7], species=[7], adata_batch_id=[7], x=[110278, 1000], y=[110278, 41], edge_index=[2, 1141520], encoder_condition_dim=0, spatial_prior_feature_dim=0, attr_decoder_condition_dim=0, adj_decoder_condition_dim=0, num_features=1000, num_classes=41, num_nodes=110278, num_edges=[7], train_mask=[110278], val_mask=[110278], test_mask=[110278], batch=[110278], ptr=[8], adata_batch_ids=[110278])
Batch ID(s): [2, 11, 9, 28, 29, 32, 12]
Data Batch: DataBatch(y_cell_types=[110278, 41], y_niche_types=[110278

In [58]:
adata_list = []
for batch in datamodule_batch.train_dataloader():
    a = pyg_batch_to_anndata(batch)
    adata_list.append(a)
    
adata = ad.concat(
    adata_list,
    axis=0,           # stack rows (cells)
    join="outer",
)

adata

AnnData object with n_obs × n_vars = 109547 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [59]:
adata.write("./xhs1000-39b_1p-oriented-7_train.h5ad")

In [60]:
adata_list = []
for batch in datamodule_batch.test_dataloader():
    a = pyg_batch_to_anndata(batch)
    adata_list.append(a)
    
adata = ad.concat(
    adata_list,
    axis=0,           # stack rows (cells)
    join="outer",
)

adata

AnnData object with n_obs × n_vars = 731 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [61]:
adata.write("./xhs1000-39b_1p-oriented-7_test.h5ad")

In [13]:
wfm_test_fname = "/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/analysis/notebooks/xhs1000-39b_1p-oriented-7_wfm_test.pkl"
with open(wfm_test_fname, "rb") as f:
    adata_wfm_test = pickle.load(f)
adata_wfm_test

/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.7.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


AnnData object with n_obs × n_vars = 731 × 1000
    obs: 'batch', 'cell_type'
    uns: 'log1p', 'pc_transform', 'niche_scale'
    obsm: 'spatial', 'X_pca', 'niche_mat', 'orig_niche_mat', 'wfm_niche_mat'

In [14]:
compute_mmd_score(
            D=[row for row in adata_wfm_test.obsm['orig_niche_mat']],
            D_hat=[row for row in adata_wfm_test.obsm['wfm_niche_mat']],
            method='scipy',
            kernel='l1_gaussian_tv',
        )

0.0875659837182609

In [15]:
compute_mmd_score(
            D=[row for row in adata_wfm_test.obsm['orig_niche_mat']],
            D_hat=[row for row in adata_wfm_test.obsm['wfm_niche_mat']],
            method='scipy',
            kernel='energy',
        )

0.1094196716908673

# xhk1020-CV1-CV2-5b_1p

In [4]:
wandb_run_dir = "/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhk1020-CV1-CV2-5b_1p/sweep/VQNiche/batch=[0, 1, 2, 3, 4]/spatial_n_neighs_8/seed/20250918-142428/wandb/run-20250918_142429-aux6obdu"

config = collect_test_configs(wandb_run_dir = wandb_run_dir)

# --------------------- Determinism Settings ---------------------
pl.seed_everything(config['experiment']['seed'])

# --------------------- Dataset ---------------------
dataset_blob = initialize_dataset_blob(config)
with open(Path(dataset_blob.processed_dir) / 'label_categories.pkl', 'rb') as f:
    label_categories = pickle.load(f)

# --------------------- Databatch ---------------------
data_batch = initialize_databatch(
                config=config,
                dataset_blob=dataset_blob,
            )

# --------------------- Dataloader ---------------------
datamodule_batch = initialize_datamodule(
                        config=config,
                        data=data_batch,

                    )


Seed set to 0


Best checkpoint found: epoch=1-train_pearson_1hop_nbr=0.92.ckpt
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
DataBatch(y_grade=[260193, 2], y_clinical_risk=[260193, 2], xy_coordinates=[260193, 2], cell_id=[5], dataset_id=[5], tissue=[5], species=[5], adata_batch_id=[5], x=[260193, 1000], y=[260193, 2], edge_index=[2, 2673293], encoder_condition_dim=0, spatial_prior_feature_dim=0, attr_decoder_condition_dim=0, adj_decoder_condition_dim=0, num_features=1000, num_classes=2, num_nodes=260193, num_edges=[5], train_mask=[260193], val_mask=[260193], test_mask=[260193], batch=[260193], ptr=[6], adata_batch_ids=[260193])
Batch ID(s): [0, 1, 2, 3, 4]
Data Batch: DataBatch(y_grade=[260193, 2], y_clinical_risk=[260193, 2], xy_coordinates=[260193, 2], cell_id=[5], dataset_id=[5], tissue=[5], species=[5], adata_batch_id=[5], x=

In [21]:
adata_list = []
for batch in datamodule_batch.predict_dataloader():
    a = pyg_batch_to_anndata(batch)
    adata_list.append(a)
    
adata = ad.concat(
    adata_list,
    axis=0,           # stack rows (cells)
    join="outer",
)

adata

AnnData object with n_obs × n_vars = 260193 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [22]:
adata.obs['grade'] = adata.obs['cell_type']
del adata.obs['cell_type']
adata

AnnData object with n_obs × n_vars = 260193 × 1000
    obs: 'batch', 'grade'
    obsm: 'spatial'

In [23]:
adata.obs['batch']

input_id
tensor(0)         0
tensor(1)         0
tensor(2)         0
tensor(3)         0
tensor(4)         0
                 ..
tensor(260188)    4
tensor(260189)    4
tensor(260190)    4
tensor(260191)    4
tensor(260192)    4
Name: batch, Length: 260193, dtype: int64

In [24]:
adata.obs['batch'].unique()

array([0, 1, 2, 3, 4])

In [13]:
# Also load original AnnData
orig_adata_list = []
dir_path = Path("/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhk1020-CV1-CV2-5b_1p")
for fname in list(dir_path.glob('**/*.h5ad')):
    print(fname)
    orig_adata_list.append(sc.read_h5ad(fname))
orig_adata_list_ordered = [orig_adata_list[2]] + [orig_adata_list[0]] + [orig_adata_list[3]] + [orig_adata_list[1]] + [orig_adata_list[4]]
orig_adata = sc.concat(orig_adata_list_ordered)
orig_adata

/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhk1020-CV1-CV2-5b_1p/adata_batch1.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhk1020-CV1-CV2-5b_1p/adata_batch3.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhk1020-CV1-CV2-5b_1p/adata_batch0.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhk1020-CV1-CV2-5b_1p/adata_batch2.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhk1020-CV1-CV2-5b_1p/adata_batch4.h5ad


/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 260193 × 4949
    obs: 'batch', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'outlier', 'grade', 'clinical_risk', 'patient', 'n_genes', 'original_cell_id'
    obsm: 'spatial'

In [26]:
mapping = {}
for orig_a in orig_adata_list:
    orig_batch = orig_a.obs['batch'].unique().tolist()[0]
    # print(orig_a.shape, orig_batch)
    for i in adata.obs['batch'].unique():
        batch_adata = adata[adata.obs['batch']==i]
        # print(batch_adata.shape, i)
        if orig_a.shape[0] == batch_adata.shape[0]:
            # print(orig_batch, i)
            mapping[orig_a.obs['batch'].unique().tolist()[0]] = i

mapping

{'Core2': 1, 'Core4': 3, 'Core1': 0, 'Core3': 2, 'Core5': 4}

In [28]:
# Load annotated AnnData object from MintFlow
adata_annot = sc.read_h5ad('/nfs/team361/aa36/OnGit/inflow-reproducibility/Analysis/42_Kidney_Rerun_NewSubPopulation/NonGit/Data/inflow_testdata_RCC3_processed_5K.h5ad')
adata_annot

AnnData object with n_obs × n_vars = 71104 × 5001
    obs: 'batch', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'outlier', '_scvi_batch', '_scvi_labels', 'SCVI_CLUSTERS_KEY', 'cell_area_highlight', 'low_quality_cell_highlight', 'level_1_cell_type', 'level_2_cell_type', 'level_3_cell_type', 'level_2_cell_type_formintflow', 'TissueSectionID_for_MintFlow', 'batchID_for_MintFlow'
    obsm: 'X_scVI', 'X_umap', 'spatial'
    layers: 'X_by_Dan'

In [9]:
adata.write("./xhk1020-CV1-CV2-5b_1p_train.h5ad")

In [10]:
adata_list = []
for batch in datamodule_batch.test_dataloader():
    a = pyg_batch_to_anndata(batch)
    adata_list.append(a)
    
adata = ad.concat(
    adata_list,
    axis=0,           # stack rows (cells)
    join="outer",
)

adata

AnnData object with n_obs × n_vars = 2447 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [11]:
adata.write("./xhk1020-CV1-CV2-5b_1p_test.h5ad")

In [12]:
wfm_test_fname = "/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/analysis/notebooks/xhk1020-CV1-CV2-5b_1p_wfm_test.pkl"
with open(wfm_test_fname, "rb") as f:
    adata_wfm_test = pickle.load(f)
adata_wfm_test

FileNotFoundError: [Errno 2] No such file or directory: '/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/analysis/notebooks/xhk1020-CV1-CV2-5b_1p_wfm_test.pkl'

In [ ]:
compute_mmd_score(
            D=[row for row in adata_wfm_test.obsm['orig_niche_mat']],
            D_hat=[row for row in adata_wfm_test.obsm['wfm_niche_mat']],
            method='scipy',
            kernel='l1_gaussian_tv',
        )

In [ ]:
compute_mmd_score(
            D=[row for row in adata_wfm_test.obsm['orig_niche_mat']],
            D_hat=[row for row in adata_wfm_test.obsm['wfm_niche_mat']],
            method='scipy',
            kernel='energy',
        )